simulation using a fixed transmissibility - beta

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import covasim as cv
from Bio import SeqIO

from covasim.jhk_covasim4wastewater.intervention_search.simulation.runner import run_evo_simulation
from covasim.jhk_covasim4wastewater.intervention_search.analysis.mutation_tree import build_mutation_tree, build_infection_seq_tree, extract_sequenced_subtree, add_internal_leaves, mutation_tree_to_newick, export_fasta_from_G, set_all_branch_lengths_to_one
from covasim.jhk_covasim4wastewater.intervention_search.plotting.plot_mutation_tree import plot_mutation_tree, plot_erase_mutation_tree, plot_phylo_tree

# from covasim.transmission_tree import TransmissionTree
# from covasim.metrics import Metrics
from covasim.inference import Inference

Covasim 3.1.8 (2026-05-29) — © 2020-2026 by IDM


In [2]:
REF_FASTA     = 'sars-2-data/NC_045512_Hu-1.fasta'
FITNESS_CSV   = '../../data/nt_fitness.csv'
ARTIC_PRIMERS = 'sars-2-data/ARTIC_V4-1.bed'
FIG_DIR = "/home/yutianc/covasim4wastewater/docs/tutorials/results"

ay4_fasta   = 'sars-2-data/variants/AY.4_consensus.fasta'
ba2_fasta   = 'sars-2-data/variants/BA.2.9_consensus.fasta'
xbb15_fasta = 'sars-2-data/variants/XBB.1.5.18_consensus.fasta'


In [3]:
ay4 = cv.variant(
    variant        = {},
    label          = 'AY.4',
    days           = 30,
    n_imports      = 60,
    founding_fasta = ay4_fasta,
)
ba2 = cv.variant(
    variant        = {},
    label          = 'BA.2',
    days           = 70,
    n_imports      = 60,
    founding_fasta = ba2_fasta,
)
xbb15 = cv.variant(
    variant        = {},
    label          = 'XBB.1.5',
    days           = 30,
    n_imports      = 60,
    founding_fasta = xbb15_fasta,
)

sample_days = list(range(10, 121, 10))
# Routine clinical: 50 symptomatic agents per day, every 10 days.
cs_routine = cv.ClinicalSequencer(
    days      = sample_days,
    n_samples = 50,
    pool      = 'symptomatic',
    label     = 'CS_routine',
    # sequencing_batch_size defaults to 1, causing immediate release
)

In [4]:
evo_pars = dict(
    enable            = False,
    reference         = REF_FASTA,
    mol_clock_rate    = 7.2e-5, # ~1 substitution/genome/month at 30 000 nt
    sub_model         = 'JC', # decides which mutations happen
    fitness_model = "bloom_nt", # controls transmission rate
    fitness_data_path = FITNESS_CSV,
    store_infection_log=True,
    fitness_scale     = 0.1,
)
sim_pars = dict(
    pop_size=100,
    pop_type='hybrid',
    pop_infected=1,
    start_day='2021-01-01', 
    n_days=121,
    beta=0.1,
    rand_seed=42,
    verbose=0,    
    variants=[xbb15],
    analyzers=[cs_routine],    
)
policy_pars = dict(
    p_test=0.3,
    p_seq=0.5,
    detection_threshold=3,
    delay_pmf=[0.05, 0.15, 0.30, 0.25, 0.15, 0.07, 0.03],
    quiet_period=0,
)
sim, obs, interv, _, results = run_evo_simulation(sim_pars, policy_pars)

cs_routine = sim.get_analyzer('CS_routine')

TypeError: run_evo_simulation() missing 1 required positional argument: 'policy'